# 05 — Score et classement sous budget

Entrées : `carreaux`, `offre_existante` (Files/gold, notebook 04).
Sorties : tables `candidats_scores`, `recommandations`, `modele_eval`, `couts`, `hypotheses` + export JSON.

In [ ]:
BUDGETS_EUR = [100_000, 250_000, 500_000, 1_000_000]
TYPES = ["citiz", "velo", "bus"]
RAYON_M = 400
ANNEES = 3
BONUS_EQUITE = 0.5
POP_MIN = 100
DIST_LIGNE_BUS_M = 800
VOITURES_EVITEES_PAR_CITIZ = 8

COUTS = {
    "citiz": {"investissement": 25_000, "annuel": 3_000},
    "velo": {"investissement": 0, "annuel": 15 * 3_000},
    "bus": {"investissement": 15_000, "annuel": 0},
}


In [ ]:
import heapq
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, cross_val_predict

GOLD = Path("/lakehouse/default/Files/gold")
car = pd.read_parquet(GOLD / "carreaux.parquet")
offre = pd.read_parquet(GOLD / "offre_existante.parquet")
car = car[car["population"] > 0].reset_index(drop=True)
print(f"{len(car):,} carreaux habités | {len(offre):,} stations/arrêts")


## Variables par carreau

In [ ]:
pts = car[["x", "y"]].to_numpy()
arbre = cKDTree(pts)
voisins = [np.asarray(v, dtype=int) for v in arbre.query_ball_point(pts, RAYON_M)]
pop = car["population"].to_numpy(float)

bdx = car["code_commune"] == "33063"
XC = np.average(car.loc[bdx, "x"], weights=car.loc[bdx, "population"])
YC = np.average(car.loc[bdx, "y"], weights=car.loc[bdx, "population"])
men = car["men"].where(car["men"] > 0)

X = pd.DataFrame({
    "pop_400m": [pop[v].sum() for v in voisins],
    "pop_carreau": pop,
    "part_collectif": car["men_coll"] / men,
    "part_proprietaires": car["men_prop"] / men,
    "part_pauvres": car["men_pauv"] / men,
    "part_65_plus": (car["ind_65_79"] + car["ind_80p"]) / car["population"],
    "dist_centre_km": np.hypot(car["x"] - XC, car["y"] - YC) / 1000,
})
for t in ("tram", "bus", "velo", "citiz", "metro", "bateau"):
    if f"nb_{t}" in car:
        X[f"nb_{t}"] = car[f"nb_{t}"]

def nb(t):
    return car[f"nb_{t}"].to_numpy() if f"nb_{t}" in car else np.zeros(len(car), int)


## Modèle d'adéquation (prédictif)

In [ ]:
groupes = (car["x"] // 2000).astype(int).astype(str) + "_" + (car["y"] // 2000).astype(int).astype(str)
n_plis = min(5, groupes.nunique())
adequation, evals = {}, []

for t in TYPES:
    stations_t = offre.loc[offre["type"] == t, ["x", "y"]].to_numpy()
    if len(stations_t) < 10:
        adequation[t] = np.full(len(car), 0.5)
        evals.append({"type": t, "n_carreaux": len(car), "n_positifs": len(stations_t), "auc_cv_spatiale": None})
        continue
    y = cKDTree(stations_t).query(pts)[0] <= 200
    feats = X.drop(columns=[f"nb_{t}"], errors="ignore")
    clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=0)
    proba = cross_val_predict(clf, feats, y, groups=groupes, cv=GroupKFold(n_plis), method="predict_proba")[:, 1]
    adequation[t] = proba
    evals.append({"type": t, "n_carreaux": len(car), "n_positifs": int(y.sum()),
                  "auc_cv_spatiale": round(roc_auc_score(y, proba), 3)})

modele_eval = pd.DataFrame(evals)
print(modele_eval.to_string(index=False))


## Candidats

In [ ]:
poids = pop * np.where(car["nb_options"].to_numpy() == 0, 1 + BONUS_EQUITE, 1.0)
couvert0 = {t: nb(t) > 0 for t in TYPES}
cout = {t: COUTS[t]["investissement"] + ANNEES * COUTS[t]["annuel"] for t in TYPES}

stops_bus = offre.loc[offre["type"] == "bus", ["x", "y"]].to_numpy()
dist_bus = cKDTree(stops_bus).query(pts)[0] if len(stops_bus) else np.full(len(car), np.inf)

lignes = []
for t in TYPES:
    facteur = 0.5 + 0.5 * pd.Series(adequation[t]).rank(pct=True).to_numpy()
    for i in np.flatnonzero(~couvert0[t]):
        if t == "bus" and dist_bus[i] > DIST_LIGNE_BUS_M:
            continue
        v = voisins[i]
        libres = v[~couvert0[t][v]]
        pop_nouvelle = pop[libres].sum()
        if pop_nouvelle < POP_MIN:
            continue
        lignes.append((t, i, cout[t], facteur[i], adequation[t][i], pop_nouvelle, poids[libres].sum() * facteur[i]))

cands = pd.DataFrame(lignes, columns=["type", "i", "cout_eur", "facteur", "adequation", "pop_nouvelle", "valeur"])
cands["id_candidat"] = cands["type"] + "_" + car["id_carreau"].to_numpy()[cands["i"]]
cands["lat"] = car["lat"].to_numpy()[cands["i"]]
cands["lon"] = car["lon"].to_numpy()[cands["i"]]
cands["code_commune"] = car["code_commune"].to_numpy()[cands["i"]]
cands["valeur_par_euro"] = cands["valeur"] / cands["cout_eur"]
VAL_MAX = cands["valeur"].max()
cands["score"] = (100 * cands["valeur"] / VAL_MAX).round(1)
print(cands.groupby("type").agg(n=("i", "size"), pop_max=("pop_nouvelle", "max"), cout=("cout_eur", "first")).to_string())


## Classement sous budget

In [ ]:
C_TYPE = cands["type"].to_numpy()
C_I = cands["i"].to_numpy()
C_COUT = cands["cout_eur"].to_numpy(float)
C_FACT = cands["facteur"].to_numpy()

def classer(budget, types=TYPES):
    couv = {t: couvert0[t].copy() for t in TYPES}
    tas = [(-cands.at[k, "valeur_par_euro"], k) for k in cands.index[cands["type"].isin(types)]]
    heapq.heapify(tas)
    reste, choix = float(budget), []
    while tas:
        _, k = heapq.heappop(tas)
        if C_COUT[k] > reste:
            continue
        t, v = C_TYPE[k], voisins[C_I[k]]
        libres = v[~couv[t][v]]
        gain = poids[libres].sum() * C_FACT[k]
        if gain <= 0:
            continue
        if tas and gain / C_COUT[k] < -tas[0][0]:
            heapq.heappush(tas, (-gain / C_COUT[k], k))
            continue
        couv[t][libres] = True
        reste -= C_COUT[k]
        choix.append((k, pop[libres].sum(), gain))
    res = cands.loc[[k for k, _, _ in choix], ["id_candidat", "type", "lat", "lon", "code_commune", "cout_eur", "adequation"]]
    res = res.assign(pop_nouvelle=[int(round(p)) for _, p, _ in choix],
                     score=[round(100 * g / VAL_MAX, 1) for _, _, g in choix]).reset_index(drop=True)
    res.insert(0, "rang", np.arange(1, len(res) + 1))
    res["cout_cumule_eur"] = res["cout_eur"].cumsum()
    res["voitures_evitees"] = np.where(res["type"] == "citiz", VOITURES_EVITEES_PAR_CITIZ, 0)
    res["budget_scenario"] = budget
    return res, couv

PERIMETRES = {"tous": TYPES, "citiz": ["citiz"]}
recommandations, impacts = [], []
pop_tot = pop.sum()
for perimetre, types in PERIMETRES.items():
  for b in BUDGETS_EUR:
    res, couv = classer(b, types)
    res["perimetre"] = perimetre
    recommandations.append(res)
    impacts.append({"perimetre": perimetre, "budget_eur": b, "nb_equipements": len(res), "cout_eur": int(res["cout_eur"].sum()),
                    "habitants_nouvellement_servis": int(res["pop_nouvelle"].sum()),
                    "voitures_evitees": int(res["voitures_evitees"].sum()),
                    **{f"part_pop_{t}_avant": round(100 * pop[couvert0[t]].sum() / pop_tot, 1) for t in TYPES},
                    **{f"part_pop_{t}_apres": round(100 * pop[couv[t]].sum() / pop_tot, 1) for t in TYPES}})
recommandations = pd.concat(recommandations, ignore_index=True)
impacts = pd.DataFrame(impacts)
print(impacts.T.to_string())
print(recommandations.query("perimetre == 'citiz' and budget_scenario == @BUDGETS_EUR[-1]").head(15).to_string(index=False))


## Écriture

In [ ]:
couts = pd.DataFrame([{"type": t, **COUTS[t], "annees": ANNEES, "cout_total_eur": cout[t]} for t in TYPES])

hyp_05 = pd.DataFrame([
    ("cout_citiz", f"{COUTS['citiz']['investissement']} € + {COUTS['citiz']['annuel']} €/an",
     "Boîtier 4 000 € et 3 000 €/an/véhicule : convention Citiz Genevois français 2023 ; véhicule ~20 000 € et signalisation : estimation à confirmer avec Citiz"),
    ("cout_velo", f"{COUTS['velo']['annuel']} €/an (15 vélos x 3 000 €)", "2 000 à 4 000 €/vélo/an (compilation Le Monde 2015, reprise par We Demain)"),
    ("cout_bus", f"{COUTS['bus']['investissement']} €", "Arrêt accessible 10 700 à 23 000 € (Cerema, rapport quais modulaires)"),
    ("horizon_cout", f"{ANNEES} ans", "Choix d'équipe"),
    ("candidats", f"centres de carreaux habités, >= {POP_MIN} nouveaux habitants à {RAYON_M} m", "Choix d'équipe"),
    ("candidats_bus", f"à moins de {DIST_LIGNE_BUS_M} m d'un arrêt existant (ligne proche)", "Choix d'équipe"),
    ("valeur", "habitants non couverts à 400 m x (0,5 + 0,5 x rang d'adéquation)", "Modèle"),
    ("bonus_equite", f"+{int(100 * BONUS_EQUITE)} % pour les habitants sans aucune option", "Choix d'équipe (équité)"),
    ("adequation", "gradient boosting, cible = station existante à 200 m, validation croisée spatiale (blocs 2 km)", "Modèle"),
    ("voitures_evitees", f"{VOITURES_EVITEES_PAR_CITIZ} par voiture Citiz", "Livre blanc autopartage Citiz 2025"),
    ("classement", "glouton paresseux (valeur ajoutée par euro), couverture mise à jour à chaque choix", "Méthode"),
], columns=["cle", "valeur", "source"])

hyp_path = GOLD / "hypotheses.parquet"
hyp_04 = pd.read_parquet(hyp_path) if hyp_path.exists() else pd.DataFrame(columns=hyp_05.columns)
hypotheses = pd.concat([hyp_04[~hyp_04["cle"].isin(hyp_05["cle"])], hyp_05], ignore_index=True)

candidats_scores = cands[["id_candidat", "type", "lat", "lon", "code_commune", "cout_eur",
                          "pop_nouvelle", "adequation", "score", "valeur_par_euro"]]

TABLES = {"candidats_scores": candidats_scores, "recommandations": recommandations, "impacts": impacts,
          "modele_eval": modele_eval, "couts": couts, "hypotheses": hypotheses}

import pyarrow as pa
from deltalake import write_deltalake
import notebookutils
ctx = notebookutils.runtime.context
DELTA = (f"abfss://{ctx['defaultLakehouseWorkspaceId']}@onelake.dfs.fabric.microsoft.com/"
         f"{ctx['defaultLakehouseId']}/Tables" + ("/dbo" if Path("/lakehouse/default/Tables/dbo").exists() else ""))
OPTIONS = {"bearer_token": notebookutils.credentials.getToken("storage"), "use_fabric_endpoint": "true"}

for nom_t, df in TABLES.items():
    df.to_parquet(GOLD / f"{nom_t}.parquet", index=False)
    write_deltalake(f"{DELTA}/{nom_t}", pa.Table.from_pandas(df, preserve_index=False),
                    mode="overwrite", schema_mode="overwrite", storage_options=OPTIONS)
    print(f"{nom_t} : {len(df):,} lignes")

EXPORT = GOLD / "export"
EXPORT.mkdir(exist_ok=True)
for nom_t in ("recommandations", "impacts", "couts", "hypotheses", "modele_eval"):
    (EXPORT / f"{nom_t}.json").write_text(TABLES[nom_t].to_json(orient="records", force_ascii=False))
